<a href="https://colab.research.google.com/github/programadormovel/13-Interface-Exemplos/blob/main/artigo003_gloss2text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Instalação

In [2]:
!pip install -q transformers datasets sentencepiece sacrebleu evaluate jiwer accelerate

2. Imports

In [3]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split

from datasets import Dataset, DatasetDict
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate

3. Carregar o CSV

Ajuste os nomes das colunas conforme seu arquivo real.

In [4]:
csv_path = "/content/pt_br2libras_gloss.csv"
df = pd.read_csv(csv_path)

print(df.head())
print(df.columns)
print(len(df))

                                               pt-br  \
0  Orientações específicas para solicitações com ...   
1  Qualquer cidadão interessado em visitar a Bibl...   
2  Qualquer cidadão interessado em doar obras par...   
3                       Qualquer cidadão interessado   
4  Qualquer cidadão e/ou empresa brasileira, com ...   

                                        libras-gloss  is_government_source  \
0  ORIENTAÇÃO ESPECÍFICO SOLICITAR OBJETIVO CONTR...                  True   
1  QUALQUER CIDADÃO INTERESSAR VISITAR BIBLIOTECA...                  True   
2  QUALQUER CIDADÃO INTERESSAR DOAR&OBJETO OBRA&L...                  True   
3                        QUALQUER CIDADÃO INTERESSAR                  True   
4  QUALQUER CIDADÃO OU EMPRESA BRASIL&PAÍS CPF OU...                  True   

                                 english_translation  
0  Specific guidelines for requests for the purpo...  
1  Any citizen interested in visiting the Zenaide...  
2  Any citizen interested in 

4. Selecionar colunas

Exemplo supondo colunas gloss e text:

In [5]:
df = df[["libras-gloss", "pt-br"]].dropna().copy()
df.columns = ["source", "target"]

Se no seu CSV os nomes forem diferentes, basta trocar.

5. Funções de normalização

In [6]:
def normalize_gloss(text):
    text = str(text).strip().lower()

    # padroniza marcações em colchetes para tokens explícitos
    text = re.sub(r"\[([^\]]+)\]", lambda m: f" <{m.group(1).strip().lower()}> ", text)

    # remove espaços duplicados
    text = re.sub(r"\s+", " ", text).strip()

    return text

def normalize_portuguese(text):
    text = str(text).strip()

    # limpeza leve
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["source"] = df["source"].apply(normalize_gloss)
df["target"] = df["target"].apply(normalize_portuguese)

6. Filtros básicos

Você pode remover linhas muito curtas ou muito longas.

In [7]:
df = df[
    (df["source"].str.len() > 1) &
    (df["target"].str.len() > 1)
].copy()

print("Total após limpeza:", len(df))

Total após limpeza: 127347


7. Divisão treino/validação/teste

Uma divisão coerente para o Colab:

80% treino
10% validação
10% teste

In [8]:
train_df, temp_df = train_test_split(df, test_size=0.10, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(len(train_df), len(val_df), len(test_df))

114612 6367 6368


8. Converter para Hugging Face Datasets

In [9]:
train_ds = Dataset.from_pandas(train_df[["source", "target"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["source", "target"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["source", "target"]], preserve_index=False)

dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['source', 'target'],
        num_rows: 114612
    })
    validation: Dataset({
        features: ['source', 'target'],
        num_rows: 6367
    })
    test: Dataset({
        features: ['source', 'target'],
        num_rows: 6368
    })
})

Tokenização
9. Carregar tokenizer e modelo pai

In [10]:
model_name = "Helsinki-NLP/opus-mt-en-es"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


10. Função de tokenização

In [11]:
max_source_length = 128
max_target_length = 128

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["source"],
        max_length=max_source_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/114612 [00:00<?, ? examples/s]

Map:   0%|          | 0/6367 [00:00<?, ? examples/s]

Map:   0%|          | 0/6368 [00:00<?, ? examples/s]

Treinamento
11. Métrica BLEU

In [12]:
bleu = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # substitui -100 pelo pad_token_id para decodificar
    labels = [[(token if token != -100 else tokenizer.pad_token_id) for token in label] for label in labels]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = bleu.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

12. Data collator

In [13]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

13. Argumentos de treinamento

No Colab, uma configuração segura pode ser:

In [14]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/gloss2pt-marian",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=30,
    predict_with_generate=True,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    report_to="none",
    gradient_accumulation_steps=2
)

Se faltar memória:

reduzir batch para 4;
aumentar gradient_accumulation_steps.

14. Trainer

In [17]:
from torch.optim import AdamW

# Move model to the correct device before initializing the optimizer
# training_args.device will be set by accelerate (e.g., 'xla' for TPUs)
model.to(training_args.device)
optimizer = AdamW(model.parameters(), lr=training_args.learning_rate, fused=False)

In [19]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None) # O segundo elemento é para o scheduler, se não for personalizado, use None
)

15. Treinar

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


16. Avaliar no conjunto de teste

In [ ]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)

Inferência
17. Função de tradução

In [ ]:
def translate_gloss(text, max_new_tokens=128):
    normalized = normalize_gloss(text)
    inputs = tokenizer(normalized, return_tensors="pt", truncation=True).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

Exemplo:


In [ ]:
sample = "PESSOA_MORA_OSASCO [INTERROGAÇÃO]"
print("Entrada:", sample)
print("Saída:", translate_gloss(sample))

Pós-processamento do português

Seu método diz que após o treinamento é necessária a normalização dos textos traduzidos, para inserir elementos da língua portuguesa e melhorar a qualidade da saída.

Essa etapa pode ser implementada inicialmente com regras simples.

18. Exemplo de pós-processamento

In [ ]:
def postprocess_portuguese(text):
    text = text.strip()

    # capitalização inicial
    if text:
        text = text[0].upper() + text[1:]

    # espaço antes de pontuação
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # interrogação explícita gerada por regra
    text = text.replace("<interrogacao>", "?")
    text = text.replace("<negacao>", "não")

    # remove espaços duplicados
    text = re.sub(r"\s+", " ", text).strip()

    # garante pontuação final opcional
    if text and text[-1] not in ".!?":
        text += "."

    return text

uso:

In [ ]:
raw_output = translate_gloss("PESSOA_MORA_OSASCO [INTERROGAÇÃO]")
final_output = postprocess_portuguese(raw_output)

print("Bruto:", raw_output)
print("Final:", final_output)